## Test Code for a decsion pipeline

In [6]:
import torch
import cv2
import numpy as np
from pathlib import Path
from model import MultiInputCNN  
from change_detection import calculate_ndvi, calculate_ndwi, detect_change

In [8]:
def load_model(model_path, device):
    model = MultiInputCNN(input_channels=3, num_classes=2)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    model.to(device)
    return model

In [10]:
def predict_image(model, image_path, device):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) / 255.0
    img_tensor = img_tensor.to(device)

    with torch.no_grad():
        output = model(img_tensor)
        pred = torch.argmax(output, dim=1).item()
    return pred

In [12]:
def make_decision(pred, veg_loss_mask, water_gain_mask):
    illegal_by_model = (pred == 0)  # 0 = Garimpo
    significant_change = veg_loss_mask.any() and water_gain_mask.any()
    if illegal_by_model and significant_change:
        return "Potential illegal mining detected"
    elif illegal_by_model:
        return "Suspicious activity (model)"
    elif significant_change:
        return "Environmental change (NDVI/NDWI)"
    else:
        return "No alert"

In [14]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model("illegal_mining_model.pt", device)

    # Paths
    rgb_image = "Affiena.png"
    red_t1, nir_t1 = "bands/red/Jan_2022-June_2022_Sentinel-2_L2A_B04(Raw).png", "bands/infrared/Jan_2022-June_2022_Sentinel-2_L2A_B08(Raw).png"
    red_t2, nir_t2 = "bands/red/Jan_2025-June_2025_Sentinel-2_L2A_B04(Raw).png", "bands/infrared/Jan_2025-June_2025_Sentinel-2_L2A_B08(Raw).png"
    green_t1, green_t2 = "bands/green/Jan_2022-June_2022__Sentinel-2_L2A_B03(Raw).png", "bands/green/Jan_2025-June_2025_Sentinel-2_L2A_B03(Raw).png"

    ##location info
    location_name = "Affiena"
    coordinates = (5.8840, -2.4691) #lat, lon

    pred = predict_image(model, rgb_image, device)

    ndvi_t1 = calculate_ndvi(red_t1, nir_t1)
    ndvi_t2 = calculate_ndvi(red_t2, nir_t2)
    ndwi_t1 = calculate_ndwi(green_t1, nir_t1)
    ndwi_t2 = calculate_ndwi(green_t2, nir_t2)

    veg_loss, water_gain, _, _ = detect_change(ndvi_t1, ndvi_t2, ndwi_t1, ndwi_t2)

    decision = make_decision(pred, veg_loss, water_gain)
    print(f"Location: {location_name} | Coordinates: {coordinates}")
    print(f"Prediction: {pred} | Decision: {decision}")
    #print("Decision:", decision)

C:\Users\aduko\AppData\Local\Temp\ipykernel_18760\1549088740.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de

Location: Affiena | Coordinates: (5.884, -2.4691)
Prediction: 0 | Decision: Potential illegal mining detected


In [50]:
import os
import csv
from datetime import datetime

#os.makedirs("results", exist_ok=True)

#csv_path = os.path.join("results", f"{location_name.replace(' ', '_')}.csv")


#with open(csv_path, "w", newline="") as f:
#    writer = csv.writer(f)
#    writer.writerow(["Location", "Latitude", "Longitude", "Prediction", "Decision"])
#    writer.writerow([location_name, coordinates[0], coordinates[1], pred, decision])

#print(f"\nResult saved to: {csv_path}")

csv_path = "results_log.csv"
header = ["Date", "Time", "Filename", "Location", "Latitude", "Longitude", "Prediction", "Decision"]

now = datetime.now()
date = now.strftime("%Y-%m-%d")
time = now.strftime("%H:%M:%S")

# Check if the file is missing or empty
write_header = not os.path.isfile(csv_path) or os.path.getsize(csv_path) == 0

with open(csv_path, "a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(header)
    writer.writerow([date, time, rgb_image, location_name, coordinates[0], coordinates[1], pred, decision])